In [ ]:
import uproot
import pandas as pd
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# Pick a FlatCAF file

In [ ]:
CAFFilePath = "/exp/dune/data/users/jskim/DDAS2026/MicroProdN4p1_NDComplex_FHC.caf.full.spineonly.0002459.CAF.flat.root"

with uproot.open(CAFFilePath) as f:
    df_caf = f['cafTree'].arrays(library='ak')

In [ ]:
df_caf["rec.mc.nu.E"].show(5)

CAFMaker converts GENIE EventRecord into a CAF True neutrino object ([SRTrueInteraction](https://github.com/DUNE/duneanaobj/blob/v03_14_00/duneanaobj/StandardRecord/SRTrueInteraction.h)) along with many other additinal information.
This means each SRTrueInteraction has a corresponding entry from the GENIE tree, and we save the GENIE tree index into [SRTrueInteraction::genieIdx](https://github.com/DUNE/duneanaobj/blob/v03_14_00/duneanaobj/StandardRecord/SRTrueInteraction.h#L50-L53) variable.
The NuSystTree is also ordered in the same way as the input GENIE Tree, so we can run a merging between two dataframes; NuSystTree with the reweights, and CAFTree with CAF variables

Let's first print `genieIdx`:

In [ ]:
df_caf["rec.mc.nu.genieIdx"]

As you can see, each element is increasing by one, which means each of the GENIE EventRecord is converted into CAF objecet one by one.

# NuSystTree

In [ ]:
NuSystTreeFilePath = "../Tutorial_Part1/NuSystTree.root"

DialColumnName_prefix = "DUNEDAS2026ExampleReweighter_NuSystTutorial"
BranchesFromNuSyst = [
    "Enu_true",
    f"tweak_responses_{DialColumnName_prefix}_DialA",
]

with uproot.open(NuSystTreeFilePath) as f:

    # Note that this time we are using awkward array
    df_nusyst = f["events"].arrays(BranchesFromNuSyst, library="ak")


We now want to "merge" NuSyst's reweight column into the CAF dataframe. There are various ways of doing this, but here we do

1. First reshape NuSystTree to match the CAF; i.e., inject the per-spill structure
1. Then copy NuSystTree columns into CAF dataframe

The per-spill structure can be injected by first counting the number of neutrinos in each spill from the CAF dataframe:

In [ ]:
arr_NNus = ak.num(df_caf["rec.mc.nu.genieIdx"])
arr_NNus.show(5)

Then, we "reshape" the nusyst Tree, by slicing the "full" neutrino entries by `arr_NNus`:

In [ ]:
df_nusyst_per_spill = ak.unflatten(df_nusyst, arr_NNus)
df_nusyst_per_spill.show(5)

Here, we intentionally also reading neutrino energy from NuSystTree (`Enu_true`), which can be used to check the correct matching to the CAF dataframe

The copying can be done by adding a new field.
Let's first add the neutrino energy and see if the matching is done properly.

In [ ]:
df_caf["Enu_true_NuSyst"] = df_nusyst_per_spill["Enu_true"]

In [ ]:
print("# CAF Enu")
df_caf[["rec.mc.nu.E"]].show(5)
print("# NuSyst Enu")
df_caf[["Enu_true_NuSyst"]].show(5)

You should see both field showing the same numbers in same order. 
Next, let's add the reweight column.

In [ ]:
df_caf["RW_DialA"] = df_nusyst_per_spill[f"tweak_responses_{DialColumnName_prefix}_DialA"]

In [ ]:
df_caf["RW_DialA"].show(5)

# Exercise 2-1

Now you have a CAF dataframe with the reweights added from NuSystTree.
Re-do the morning session, CAF selection, but now also draw a reweighted distribution.
How does a reweight in Q2 affects the neutrino energy distribution?